# 05 — Relevant Python 3.12 Changes: Advanced Problems with Solutions

This notebook is an **advanced, problem-driven** companion to a feature walkthrough. It focuses on Python 3.12 changes that are especially useful in application code, libraries, tooling, data pipelines, and typed codebases.

Every problem includes:

- a concrete specification;
- edge cases and constraints;
- a complete reference solution;
- executable checks;
- notes about correctness, maintainability, and migration.

> **Required runtime:** Python 3.12 or newer.  
> The code intentionally uses syntax that cannot be parsed by Python 3.11 or older.

## Primary references

- [What's New in Python 3.12](https://docs.python.org/3/whatsnew/3.12.html)
- [PEP 695 — Type Parameter Syntax](https://peps.python.org/pep-0695/)
- [PEP 701 — Syntactic formalization of f-strings](https://peps.python.org/pep-0701/)
- [PEP 709 — Inlined comprehensions](https://peps.python.org/pep-0709/)
- [PEP 692 — Precise `**kwargs` typing](https://peps.python.org/pep-0692/)
- [PEP 698 — `@override`](https://peps.python.org/pep-0698/)
- [PEP 688 — Python-level buffer protocol](https://peps.python.org/pep-0688/)
- [`sys.monitoring`](https://docs.python.org/3.12/library/sys.monitoring.html)

## How to use this notebook

1. Read the feature summary and examples.
2. Attempt each problem before opening its solution mentally.
3. Run the reference solution and checks.
4. Modify tests and add adversarial cases.
5. For typing exercises, also run a static checker such as Pyright or mypy in a real project.

## Learning objectives

By the end, you should be able to:

1. use Python 3.12's expanded f-string grammar without sacrificing readability;
2. design generic functions, classes, and type aliases with PEP 695 syntax;
3. model precise keyword arguments and intentional method overrides;
4. inspect the behavioral and bytecode implications of inlined comprehensions;
5. build streaming pipelines with `itertools.batched`;
6. use `math.sumprod` for validated dot products and weighted calculations;
7. traverse and prune directory trees with `Path.walk`;
8. expose buffers from pure Python classes;
9. instrument selected functions with `sys.monitoring`;
10. identify common Python 3.12 migration issues;
11. combine several features in a tested capstone.

## 0. Environment and reusable test helpers

In [1]:
from __future__ import annotations

import ast
import dis
import hashlib
import inspect
import math
import shutil
import sys
import tempfile
import timeit
import warnings

from collections import Counter, defaultdict, deque
from collections.abc import (
    Buffer,
    Callable,
    Generator,
    Hashable,
    Iterable,
    Iterator,
    Mapping,
    Sequence,
)
from dataclasses import dataclass
from itertools import batched
from pathlib import Path
from typing import (
    Any,
    NotRequired,
    Required,
    TypedDict,
    Unpack,
    get_args,
    get_origin,
    override,
)

assert sys.version_info >= (3, 12), (
    "This notebook requires Python 3.12+. "
    f"Current runtime: {sys.version.split()[0]}"
)

print("Python:", sys.version.split()[0])

Python: 3.13.7


In [2]:
def check_equal(actual: object, expected: object, *, label: str = "") -> None:
    assert actual == expected, (
        f"{label + ': ' if label else ''}"
        f"expected {expected!r}, got {actual!r}"
    )


def check_close(
    actual: float,
    expected: float,
    *,
    rel_tol: float = 1e-12,
    abs_tol: float = 0.0,
    label: str = "",
) -> None:
    assert math.isclose(actual, expected, rel_tol=rel_tol, abs_tol=abs_tol), (
        f"{label + ': ' if label else ''}"
        f"expected approximately {expected!r}, got {actual!r}"
    )


def check_raises(
    exception_type: type[BaseException],
    func: Callable[..., object],
    /,
    *args: object,
    contains: str | None = None,
    **kwargs: object,
) -> BaseException:
    try:
        func(*args, **kwargs)
    except exception_type as exc:
        if contains is not None:
            assert contains in str(exc), (
                f"expected {contains!r} in exception message {str(exc)!r}"
            )
        return exc
    except BaseException as exc:
        raise AssertionError(
            f"expected {exception_type.__name__}, got {type(exc).__name__}"
        ) from exc
    raise AssertionError(f"expected {exception_type.__name__} to be raised")

# 1. PEP 701 — More expressive f-strings

Python 3.12 integrates f-strings into the grammar and removes several historical restrictions.

Useful consequences include:

- reusing the same quote character inside replacement expressions;
- placing comments and multi-line expressions inside replacement fields;
- nesting f-strings more naturally;
- receiving parser-quality syntax errors for malformed f-strings.

## Examples

In [3]:
record = {"name": "Ada", "scores": [98, 91, 95]}

# Reusing the outer quote style inside the expression is valid in Python 3.12+.
message = f"{record["name"]} averaged {sum(record["scores"]) / len(record["scores"]):.2f}"
print(message)

# A multi-line replacement expression may contain comments.
summary = f"""Best score: {
    max(
        record["scores"]  # ignore absent assignments in a real pipeline
    )
}"""
print(summary)

# Nested formatting: the precision is itself selected dynamically.
precision = 3
value = 12.34567
print(f"{value:.{precision}f}")

Ada averaged 94.67
Best score: 98
12.346


## Problem 1 — Build a robust metric formatter

Implement `format_metric` with this behavior:

```python
format_metric(
    name="latency",
    value=12.34567,
    unit="ms",
    width=18,
    precision=2,
    warning_above=10,
)
```

Expected result:

```text
"latency      12.35 ms !"
```

Requirements:

1. `name` is left-aligned.
2. The numeric field is right-aligned.
3. `precision` controls decimal places.
4. Append `" !"` only when `warning_above` is not `None` and `value` exceeds it.
5. Reject negative `precision`.
6. Reject widths too small to contain at least one character for the name and number.

A Python 3.12 f-string should perform the dynamic width/precision formatting.

### Suggested starting point

```python
def format_metric(...):
    # Validate first.
    # Compute the warning marker.
    # Use one dynamically parameterized f-string.
    ...
```

### Solution 1

In [4]:
def format_metric(
    *,
    name: str,
    value: float,
    unit: str = "",
    width: int = 18,
    precision: int = 2,
    warning_above: float | None = None,
) -> str:
    if precision < 0:
        raise ValueError("precision must be non-negative")
    if width < 3:
        raise ValueError("width must be at least 3")

    marker = " !" if warning_above is not None and value > warning_above else ""
    suffix = f" {unit}" if unit else ""

    # Reserve enough room for a useful numeric field.
    number_width = max(precision + 3, width // 2)
    name_width = width - number_width

    if name_width < 1:
        raise ValueError("width leaves no room for the metric name")

    return f"{name:<{name_width}}{value:>{number_width}.{precision}f}{suffix}{marker}"


print(
    format_metric(
        name="latency",
        value=12.34567,
        unit="ms",
        width=18,
        precision=2,
        warning_above=10,
    )
)

latency      12.35 ms !


In [5]:
check_equal(
    format_metric(
        name="latency",
        value=12.34567,
        unit="ms",
        width=18,
        precision=2,
        warning_above=10,
    ),
    "latency      12.35 ms !",
)
check_equal(
    format_metric(name="ok", value=1.25, width=10, precision=1),
    "ok     1.2",
)
check_raises(ValueError, format_metric, name="x", value=1, precision=-1)
check_raises(ValueError, format_metric, name="x", value=1, width=2)

ValueError('width must be at least 3')

## Problem 2 — Render an incident report without precomputed temporary strings

Create `render_incident` that returns a multi-line report.

Input:

```python
incident = {
    "service": "payments",
    "severity": 2,
    "durations": [1.2, 3.4, 2.0],
    "owner": {"name": "Mina", "team": "SRE"},
}
```

Required output shape:

```text
service=payments
owner=Mina (SRE)
severity=SEV-2
samples=3
mean=2.20s
max=3.40s
```

Requirements:

- validate that `durations` is non-empty;
- use quote reuse such as `incident["service"]`;
- use at least one multi-line f-string replacement expression;
- avoid repeatedly calculating the mean.

### Solution 2

In [6]:
type Incident = dict[str, object]


def render_incident(incident: Incident) -> str:
    durations_obj = incident.get("durations")
    if not isinstance(durations_obj, list) or not durations_obj:
        raise ValueError("durations must be a non-empty list")

    durations = [float(value) for value in durations_obj]
    mean = math.fsum(durations) / len(durations)

    owner = incident["owner"]
    if not isinstance(owner, dict):
        raise TypeError("owner must be a mapping-like dictionary")

    return f"""service={incident["service"]}
owner={owner["name"]} ({owner["team"]})
severity=SEV-{incident["severity"]}
samples={
    len(
        durations  # Python 3.12 permits comments in this replacement field.
    )
}
mean={mean:.2f}s
max={max(durations):.2f}s"""


incident = {
    "service": "payments",
    "severity": 2,
    "durations": [1.2, 3.4, 2.0],
    "owner": {"name": "Mina", "team": "SRE"},
}

print(render_incident(incident))

service=payments
owner=Mina (SRE)
severity=SEV-2
samples=3
mean=2.20s
max=3.40s


In [7]:
expected_incident_report = '''service=payments
owner=Mina (SRE)
severity=SEV-2
samples=3
mean=2.20s
max=3.40s'''

check_equal(render_incident(incident), expected_incident_report)
check_raises(
    ValueError,
    render_incident,
    {"service": "x", "severity": 1, "durations": [], "owner": {}},
    contains="non-empty",
)

ValueError('durations must be a non-empty list')

# 2. PEP 695 — Type parameter syntax and the `type` statement

Python 3.12 adds dedicated syntax for generic functions, classes, and type aliases.

Old style:

```python
T = TypeVar("T")

def first(items: Sequence[T]) -> T:
    ...
```

Python 3.12 style:

```python
def first[T](items: Sequence[T]) -> T:
    ...
```

The runtime remains dynamically typed. Type checkers enforce the intended relationships.

In [8]:
def first[T](items: Sequence[T]) -> T:
    if not items:
        raise ValueError("items must not be empty")
    return items[0]


class Pair[L, R]:
    def __init__(self, left: L, right: R) -> None:
        self.left = left
        self.right = right

    def swapped(self) -> Pair[R, L]:
        return Pair(self.right, self.left)

    def __repr__(self) -> str:
        return f"Pair(left={self.left!r}, right={self.right!r})"


type Coordinate[T: int | float] = tuple[T, T]
type RecursiveList[T] = T | list[RecursiveList[T]]

print(first(["alpha", "beta"]))
print(Pair("age", 37).swapped())
print("first type parameters:", first.__type_params__)
print("Pair type parameters:", Pair.__type_params__)
print("Coordinate alias:", Coordinate)

alpha
Pair(left=37, right='age')
first type parameters: (T,)
Pair type parameters: (L, R)
Coordinate alias: Coordinate


## Problem 3 — Generic bounded stack

Implement `BoundedStack[T]`.

Requirements:

1. Constructor accepts a positive `capacity`.
2. `push(item)` raises `OverflowError` when full.
3. `pop()` raises `IndexError` when empty.
4. `peek()` returns the top item without removal.
5. `len(stack)` returns the number of stored items.
6. Iteration yields items from top to bottom.
7. `map(func)` returns a new `BoundedStack[U]` while preserving logical order.

Do not inherit from `list`. Keep the storage private.

### Solution 3

In [9]:
class BoundedStack[T]:
    def __init__(self, capacity: int) -> None:
        if capacity <= 0:
            raise ValueError("capacity must be positive")
        self._capacity = capacity
        self._items: list[T] = []

    @property
    def capacity(self) -> int:
        return self._capacity

    def push(self, item: T) -> None:
        if len(self._items) >= self._capacity:
            raise OverflowError("stack is full")
        self._items.append(item)

    def pop(self) -> T:
        if not self._items:
            raise IndexError("pop from empty stack")
        return self._items.pop()

    def peek(self) -> T:
        if not self._items:
            raise IndexError("peek from empty stack")
        return self._items[-1]

    def __len__(self) -> int:
        return len(self._items)

    def __iter__(self) -> Iterator[T]:
        return reversed(self._items)

    def map[U](self, func: Callable[[T], U]) -> BoundedStack[U]:
        mapped = BoundedStack[U](self.capacity)
        # Insert bottom-to-top so iteration order remains top-to-bottom.
        for item in self._items:
            mapped.push(func(item))
        return mapped

    def __repr__(self) -> str:
        return (
            f"BoundedStack(capacity={self.capacity}, "
            f"top_to_bottom={list(self)!r})"
        )

In [10]:
stack = BoundedStack[int](3)
for value in (10, 20, 30):
    stack.push(value)

check_equal(stack.peek(), 30)
check_equal(list(stack), [30, 20, 10])
check_raises(OverflowError, stack.push, 40)

mapped_stack = stack.map(lambda number: f"#{number}")
check_equal(list(mapped_stack), ["#30", "#20", "#10"])
check_equal(stack.pop(), 30)
check_equal(len(stack), 2)

empty = BoundedStack[str](1)
check_raises(IndexError, empty.pop)
check_raises(IndexError, empty.peek)
check_raises(ValueError, BoundedStack, 0)

ValueError('capacity must be positive')

## Problem 4 — Generic `group_by`

Implement a reusable generic function:

```python
def group_by[T, K: Hashable](
    items: Iterable[T],
    key: Callable[[T], K],
) -> dict[K, list[T]]:
    ...
```

Requirements:

- consume the iterable exactly once;
- preserve encounter order inside each group;
- support generators;
- never assume the key is a string;
- return a plain dictionary.

### Solution 4

In [11]:
def group_by[T, K: Hashable](
    items: Iterable[T],
    key: Callable[[T], K],
) -> dict[K, list[T]]:
    groups: dict[K, list[T]] = {}

    for item in items:
        group_key = key(item)
        groups.setdefault(group_key, []).append(item)

    return groups


words = (word for word in ["pear", "plum", "apple", "apricot", "fig"])
grouped_words = group_by(words, key=lambda word: word[0])
grouped_words

{'p': ['pear', 'plum'], 'a': ['apple', 'apricot'], 'f': ['fig']}

In [12]:
check_equal(
    grouped_words,
    {
        "p": ["pear", "plum"],
        "a": ["apple", "apricot"],
        "f": ["fig"],
    },
)

check_equal(
    group_by(range(7), key=lambda number: number % 3),
    {0: [0, 3, 6], 1: [1, 4], 2: [2, 5]},
)

## Problem 5 — Recursive JSON alias and leaf traversal

Define a recursive type alias `JSONValue` and implement `walk_json_leaves`.

For this value:

```python
{
    "user": {"name": "Ada", "active": True},
    "scores": [10, 20],
}
```

yield:

```python
("user.name", "Ada")
("user.active", True)
("scores.0", 10)
("scores.1", 20)
```

Requirements:

- dictionary keys are strings;
- list positions become decimal path components;
- the empty path is represented by `"<root>"`;
- preserve dictionary insertion order and list order;
- use structural pattern matching in the solution.

### Solution 5

In [13]:
type JSONScalar = None | bool | int | float | str
type JSONValue = JSONScalar | list[JSONValue] | dict[str, JSONValue]


def walk_json_leaves(
    value: JSONValue,
    path: tuple[str, ...] = (),
) -> Iterator[tuple[str, JSONScalar]]:
    match value:
        case dict() as mapping:
            for key, child in mapping.items():
                yield from walk_json_leaves(child, (*path, key))

        case list() as sequence:
            for index, child in enumerate(sequence):
                yield from walk_json_leaves(child, (*path, str(index)))

        case None | bool() | int() | float() | str() as scalar:
            yield (".".join(path) or "<root>", scalar)

        case _:
            raise TypeError(f"unsupported JSON value: {type(value).__name__}")


sample_json: JSONValue = {
    "user": {"name": "Ada", "active": True},
    "scores": [10, 20],
}

list(walk_json_leaves(sample_json))

[('user.name', 'Ada'),
 ('user.active', True),
 ('scores.0', 10),
 ('scores.1', 20)]

In [14]:
check_equal(
    list(walk_json_leaves(sample_json)),
    [
        ("user.name", "Ada"),
        ("user.active", True),
        ("scores.0", 10),
        ("scores.1", 20),
    ],
)
check_equal(list(walk_json_leaves(42)), [("<root>", 42)])
check_raises(TypeError, lambda: list(walk_json_leaves({1, 2, 3})))  # type: ignore[arg-type]

TypeError('unsupported JSON value: set')

# 3. More precise typing — `Unpack[TypedDict]` and `@override`

Python 3.12's typing improvements are mostly checked by static analysis tools.

- PEP 692 lets `**kwargs` describe an exact keyword schema using `Unpack[TypedDict]`.
- PEP 698 adds `typing.override` to mark methods intended to override a base method.

At runtime, `@override` sets `__override__ = True` when possible. It does not itself validate inheritance.

## Problem 6 — Precisely typed request options

Implement a function accepting the following keyword-only schema:

- required: `timeout: float`;
- optional: `retries: int`;
- optional: `headers: Mapping[str, str]`;
- optional: `follow_redirects: bool`.

Return an immutable normalized tuple:

```python
(timeout, retries, sorted_header_pairs, follow_redirects)
```

Also perform runtime validation because type hints do not validate external input.

### Solution 6

In [15]:
class RequestOptions(TypedDict):
    timeout: Required[float]
    retries: NotRequired[int]
    headers: NotRequired[Mapping[str, str]]
    follow_redirects: NotRequired[bool]


type NormalizedRequestOptions = tuple[
    float,
    int,
    tuple[tuple[str, str], ...],
    bool,
]


def normalize_request_options(
    **kwargs: Unpack[RequestOptions],
) -> NormalizedRequestOptions:
    timeout = kwargs["timeout"]
    retries = kwargs.get("retries", 0)
    headers = kwargs.get("headers", {})
    follow_redirects = kwargs.get("follow_redirects", True)

    if timeout <= 0:
        raise ValueError("timeout must be positive")
    if retries < 0:
        raise ValueError("retries must be non-negative")
    if not all(isinstance(k, str) and isinstance(v, str) for k, v in headers.items()):
        raise TypeError("headers must map strings to strings")

    return (
        float(timeout),
        retries,
        tuple(sorted(headers.items())),
        follow_redirects,
    )


normalize_request_options(
    timeout=2.5,
    retries=3,
    headers={"X-Trace": "abc", "Accept": "application/json"},
)

(2.5, 3, (('Accept', 'application/json'), ('X-Trace', 'abc')), True)

In [16]:
check_equal(
    normalize_request_options(
        timeout=2.5,
        retries=3,
        headers={"X-Trace": "abc", "Accept": "application/json"},
    ),
    (
        2.5,
        3,
        (("Accept", "application/json"), ("X-Trace", "abc")),
        True,
    ),
)
check_equal(
    normalize_request_options(timeout=1),
    (1.0, 0, (), True),
)
check_raises(ValueError, normalize_request_options, timeout=0)
check_raises(ValueError, normalize_request_options, timeout=1, retries=-1)

ValueError('retries must be non-negative')

### Static-checking examples

A type checker should reject these calls:

```python
normalize_request_options(retries=2)                # missing timeout
normalize_request_options(timeout="slow")           # wrong value type
normalize_request_options(timeout=1, cache=True)    # unknown keyword
```

Runtime validation remains important when values originate from JSON, environment variables, command-line arguments, databases, or untyped code.

## Problem 7 — Intentional overrides in a renderer hierarchy

Build:

- abstract-style base class `Renderer`;
- `TextRenderer`;
- `JSONRenderer`.

Each subclass must mark its implementation with `@override`.

`Renderer.render` should raise `NotImplementedError`.  
`TextRenderer` should produce `key=value` pairs sorted by key.  
`JSONRenderer` should produce a deterministic compact JSON string without importing third-party packages.

### Solution 7

In [17]:
import json


class Renderer:
    def render(self, data: Mapping[str, object]) -> str:
        raise NotImplementedError


class TextRenderer(Renderer):
    @override
    def render(self, data: Mapping[str, object]) -> str:
        return " ".join(f"{key}={data[key]}" for key in sorted(data))


class JSONRenderer(Renderer):
    @override
    def render(self, data: Mapping[str, object]) -> str:
        return json.dumps(
            dict(data),
            sort_keys=True,
            separators=(",", ":"),
        )


payload = {"status": "ok", "count": 3}
print(TextRenderer().render(payload))
print(JSONRenderer().render(payload))
print("override marker:", TextRenderer.render.__override__)

count=3 status=ok
{"count":3,"status":"ok"}
override marker: True


In [18]:
check_equal(TextRenderer().render(payload), "count=3 status=ok")
check_equal(
    JSONRenderer().render(payload),
    '{"count":3,"status":"ok"}',
)
check_equal(getattr(TextRenderer.render, "__override__", False), True)
check_equal(getattr(JSONRenderer.render, "__override__", False), True)
check_raises(NotImplementedError, Renderer().render, payload)

NotImplementedError()

# 4. PEP 709 — Inlined comprehensions

Before Python 3.12, a comprehension was commonly compiled as a nested function-like code object. Python 3.12 inlines list, set, and dictionary comprehensions.

Practical effects include:

- less call/frame overhead;
- faster comprehensions;
- no separate comprehension frame in tracebacks;
- different introspection/tracing details;
- comprehension loop variables still do **not** leak into the surrounding scope.

In [19]:
def squares(values: Iterable[int]) -> list[int]:
    return [value * value for value in values]


nested_code_objects = [
    const
    for const in squares.__code__.co_consts
    if inspect.iscode(const)
]

print("nested code object names:", [obj.co_name for obj in nested_code_objects])
print("result:", squares(range(6)))

# In Python 3.12+, there should be no separate <listcomp> code object here.
assert "<listcomp>" not in [obj.co_name for obj in nested_code_objects]

nested code object names: []
result: [0, 1, 4, 9, 16, 25]


## Problem 8 — Normalize records with one inlined comprehension

Given records such as:

```python
[
    {"name": " Ada ", "score": 91},
    {"name": "GRACE", "score": 88},
    {"name": "", "score": 100},
    {"name": "Linus", "score": -1},
]
```

Return normalized `(name, score)` pairs where:

- names are stripped and case-folded;
- blank names are rejected;
- scores must be between 0 and 100 inclusive;
- output is sorted by descending score, then normalized name;
- the main filtering/transformation must be one comprehension;
- avoid normalizing the same name twice by using an assignment expression.

### Solution 8

In [20]:
type RawRecord = Mapping[str, object]
type NormalizedRecord = tuple[str, int]


def normalize_records(records: Iterable[RawRecord]) -> list[NormalizedRecord]:
    normalized = [
        (clean_name, score)
        for record in records
        if isinstance((raw_name := record.get("name")), str)
        if (clean_name := raw_name.strip().casefold())
        if isinstance((score := record.get("score")), int)
        if 0 <= score <= 100
    ]

    return sorted(normalized, key=lambda item: (-item[1], item[0]))


raw_records = [
    {"name": " Ada ", "score": 91},
    {"name": "GRACE", "score": 88},
    {"name": "", "score": 100},
    {"name": "Linus", "score": -1},
]

normalize_records(raw_records)

[('ada', 91), ('grace', 88)]

In [21]:
check_equal(
    normalize_records(raw_records),
    [("ada", 91), ("grace", 88)],
)

# Comprehension loop variables do not leak.
sentinel = "outside"
_ = [sentinel for sentinel in range(3)]
check_equal(sentinel, "outside")

normalize_nested_names = [
    const.co_name
    for const in normalize_records.__code__.co_consts
    if inspect.iscode(const)
]
assert "<listcomp>" not in normalize_nested_names

## Problem 9 — Compare a comprehension pipeline with an explicit loop

Implement two equivalent functions that:

1. accept integers;
2. keep only positive even values;
3. square each retained value;
4. return a list.

Then:

- verify equality over multiple inputs;
- inspect nested code objects;
- benchmark both implementations without asserting that timing is stable.

The goal is not “comprehensions are always better.” The goal is to understand the optimized execution model and choose the clearest implementation.

### Solution 9

In [22]:
def transform_comprehension(values: Iterable[int]) -> list[int]:
    return [value * value for value in values if value > 0 and value % 2 == 0]


def transform_loop(values: Iterable[int]) -> list[int]:
    result: list[int] = []
    for value in values:
        if value > 0 and value % 2 == 0:
            result.append(value * value)
    return result


for case in ([], [1], [-2, 0, 2, 3, 4], list(range(-100, 100))):
    check_equal(transform_comprehension(case), transform_loop(case))

print(
    "comprehension nested code objects:",
    [
        const.co_name
        for const in transform_comprehension.__code__.co_consts
        if inspect.iscode(const)
    ],
)

data = list(range(-1_000, 1_000))
comprehension_time = timeit.timeit(
    "transform_comprehension(data)",
    globals=globals(),
    number=2_000,
)
loop_time = timeit.timeit(
    "transform_loop(data)",
    globals=globals(),
    number=2_000,
)

print(f"comprehension: {comprehension_time:.4f}s")
print(f"explicit loop: {loop_time:.4f}s")
print(f"loop/comprehension ratio: {loop_time / comprehension_time:.2f}x")

comprehension nested code objects: []
comprehension: 0.5907s
explicit loop: 0.8295s
loop/comprehension ratio: 1.40x


# 5. `itertools.batched` — Streaming fixed-size groups

Python 3.12 adds `itertools.batched(iterable, n)`.

Important properties:

- it accepts any iterable, including generators;
- it consumes lazily;
- it yields tuples;
- the final tuple may contain fewer than `n` items;
- Python 3.12 does **not** have the later `strict=` argument.

In [23]:
print(list(batched("ABCDEFG", 3)))
check_equal(
    list(batched("ABCDEFG", 3)),
    [("A", "B", "C"), ("D", "E", "F"), ("G",)],
)
check_raises(ValueError, lambda: list(batched([1, 2], 0)))

[('A', 'B', 'C'), ('D', 'E', 'F'), ('G',)]


ValueError('n must be at least one')

## Problem 10 — Batch a stream and calculate per-batch metrics

Implement `summarize_batches`.

For each batch, return:

```python
{
    "index": 1,
    "size": 3,
    "minimum": ...,
    "maximum": ...,
    "mean": ...,
}
```

Requirements:

- accept any iterable of numbers;
- consume it once;
- reject non-positive batch sizes;
- return an empty list for empty input;
- number batches starting at 1;
- use `math.fsum` for the mean.

### Solution 10

In [24]:
class BatchSummary(TypedDict):
    index: int
    size: int
    minimum: float
    maximum: float
    mean: float


def summarize_batches(
    values: Iterable[float],
    batch_size: int,
) -> list[BatchSummary]:
    if batch_size <= 0:
        raise ValueError("batch_size must be positive")

    summaries: list[BatchSummary] = []

    for index, batch in enumerate(batched(values, batch_size), start=1):
        numeric_batch = tuple(float(value) for value in batch)
        summaries.append(
            {
                "index": index,
                "size": len(numeric_batch),
                "minimum": min(numeric_batch),
                "maximum": max(numeric_batch),
                "mean": math.fsum(numeric_batch) / len(numeric_batch),
            }
        )

    return summaries


summarize_batches((value / 2 for value in range(1, 8)), 3)

[{'index': 1, 'size': 3, 'minimum': 0.5, 'maximum': 1.5, 'mean': 1.0},
 {'index': 2, 'size': 3, 'minimum': 2.0, 'maximum': 3.0, 'mean': 2.5},
 {'index': 3, 'size': 1, 'minimum': 3.5, 'maximum': 3.5, 'mean': 3.5}]

In [25]:
batch_result = summarize_batches((value / 2 for value in range(1, 8)), 3)
check_equal(
    batch_result,
    [
        {"index": 1, "size": 3, "minimum": 0.5, "maximum": 1.5, "mean": 1.0},
        {"index": 2, "size": 3, "minimum": 2.0, "maximum": 3.0, "mean": 2.5},
        {"index": 3, "size": 1, "minimum": 3.5, "maximum": 3.5, "mean": 3.5},
    ],
)
check_equal(summarize_batches([], 4), [])
check_raises(ValueError, summarize_batches, [1, 2], 0)

ValueError('batch_size must be positive')

## Problem 11 — Add strict batching behavior for Python 3.12

Python 3.13 later added `strict=` to `batched`, but Python 3.12's API does not include it.

Implement a Python 3.12-compatible `batched_strict`:

- yield full tuples of exactly `n` items;
- raise `ValueError` if the final batch is incomplete;
- reject `n < 1`;
- remain lazy;
- avoid converting the entire iterable to a collection.

### Solution 11

In [26]:
def batched_strict[T](
    iterable: Iterable[T],
    n: int,
) -> Iterator[tuple[T, ...]]:
    if n < 1:
        raise ValueError("n must be at least one")

    for batch in batched(iterable, n):
        if len(batch) != n:
            raise ValueError(
                f"incomplete final batch: expected {n}, got {len(batch)}"
            )
        yield batch


check_equal(
    list(batched_strict(range(6), 3)),
    [(0, 1, 2), (3, 4, 5)],
)
check_raises(
    ValueError,
    lambda: list(batched_strict(range(7), 3)),
    contains="incomplete final batch",
)
check_raises(ValueError, lambda: list(batched_strict(range(3), 0)))

ValueError('n must be at least one')

# 6. `math.sumprod` and `math.nextafter(..., steps=...)`

Python 3.12 adds:

- `math.sumprod(p, q)` — a validated sum of pairwise products;
- a `steps` argument for `math.nextafter`.

`sumprod` raises `ValueError` for mismatched lengths and uses extended precision for float and mixed int/float intermediate calculations.

In [27]:
weights = [0.25, 0.35, 0.40]
scores = [80, 90, 100]

print("weighted score:", math.sumprod(weights, scores))
check_close(math.sumprod(weights, scores), 91.5)
check_raises(ValueError, math.sumprod, [1, 2], [3])

x = 1.0
print("five representable floats toward +∞:", math.nextafter(x, math.inf, steps=5))
assert math.nextafter(x, math.inf, steps=5) > x

weighted score: 91.5
five representable floats toward +∞: 1.000000000000001


## Problem 12 — Portfolio exposure with validated dimensions

Implement `portfolio_exposure`.

Inputs:

- `quantities`: units held;
- `prices`: price per unit;
- `risk_weights`: multiplicative risk factors.

Return:

```python
{
    "market_value": ...,
    "risk_adjusted_value": ...,
    "contributions": (...),
}
```

Definitions:

- contribution = `quantity * price`;
- market value = sum of contributions;
- risk-adjusted value = sum of `contribution * risk_weight`.

Requirements:

- all three inputs must have equal length;
- reject negative quantities, prices, or risk weights;
- support generators by materializing each input once;
- use `math.sumprod` for both aggregate calculations.

### Solution 12

In [28]:
class ExposureResult(TypedDict):
    market_value: float
    risk_adjusted_value: float
    contributions: tuple[float, ...]


def portfolio_exposure(
    quantities: Iterable[float],
    prices: Iterable[float],
    risk_weights: Iterable[float],
) -> ExposureResult:
    quantity_values = tuple(float(value) for value in quantities)
    price_values = tuple(float(value) for value in prices)
    risk_values = tuple(float(value) for value in risk_weights)

    if not (
        len(quantity_values) == len(price_values) == len(risk_values)
    ):
        raise ValueError("quantities, prices, and risk_weights must have equal length")

    if any(value < 0 for value in (*quantity_values, *price_values, *risk_values)):
        raise ValueError("inputs must be non-negative")

    contributions = tuple(
        quantity * price
        for quantity, price in zip(
            quantity_values,
            price_values,
            strict=True,
        )
    )

    return {
        "market_value": math.sumprod(quantity_values, price_values),
        "risk_adjusted_value": math.sumprod(contributions, risk_values),
        "contributions": contributions,
    }


portfolio_exposure(
    quantities=(10, 5, 2),
    prices=(12.5, 20.0, 100.0),
    risk_weights=(0.2, 0.5, 0.8),
)

{'market_value': 425.0,
 'risk_adjusted_value': 235.0,
 'contributions': (125.0, 100.0, 200.0)}

In [29]:
exposure = portfolio_exposure(
    quantities=(10, 5, 2),
    prices=(12.5, 20.0, 100.0),
    risk_weights=(0.2, 0.5, 0.8),
)

check_equal(exposure["contributions"], (125.0, 100.0, 200.0))
check_close(exposure["market_value"], 425.0)
check_close(exposure["risk_adjusted_value"], 235.0)
check_raises(
    ValueError,
    portfolio_exposure,
    [1, 2],
    [10],
    [0.5, 0.5],
    contains="equal length",
)
check_raises(
    ValueError,
    portfolio_exposure,
    [1],
    [-10],
    [0.5],
    contains="non-negative",
)

ValueError('inputs must be non-negative')

## Problem 13 — Linear-model prediction

Implement:

```python
predict_linear(features, coefficients, *, intercept=0.0)
```

Requirements:

- use `math.sumprod`;
- enforce equal dimensions;
- reject non-finite values (`NaN`, positive infinity, negative infinity);
- return `intercept + dot(features, coefficients)`;
- accept arbitrary iterables.

### Solution 13

In [30]:
def predict_linear(
    features: Iterable[float],
    coefficients: Iterable[float],
    *,
    intercept: float = 0.0,
) -> float:
    feature_values = tuple(float(value) for value in features)
    coefficient_values = tuple(float(value) for value in coefficients)
    intercept = float(intercept)

    all_values = (*feature_values, *coefficient_values, intercept)
    if not all(math.isfinite(value) for value in all_values):
        raise ValueError("all values must be finite")

    # sumprod performs the equal-length check.
    return intercept + math.sumprod(feature_values, coefficient_values)


check_close(
    predict_linear([2, 3, 4], [0.5, -1.0, 2.0], intercept=10),
    16.0,
)
check_raises(ValueError, predict_linear, [1, 2], [3])
check_raises(ValueError, predict_linear, [math.inf], [1], contains="finite")

ValueError('all values must be finite')

# 7. `pathlib.Path.walk` — Object-oriented directory traversal

Python 3.12 adds `Path.walk`, a `pathlib`-native alternative to `os.walk`.

It yields:

```python
(directory_path, directory_names, file_names)
```

When walking top-down, mutate `directory_names` in place to prune traversal.

## Build a temporary project tree used by the next two problems

In [31]:
PROJECT_ROOT = Path(tempfile.mkdtemp(prefix="py312_walk_demo_"))

files_to_create = {
    "README.md": b"demo project\n",
    "src/app.py": b"print('hello')\n",
    "src/utils.py": b"VALUE = 42\n",
    "src/data.bin": b"\x00\x01\x02",
    "tests/test_app.py": b"def test_ok(): assert True\n",
    "docs/guide.md": b"demo project\n",  # duplicate of README.md
    ".git/config": b"[core]\n",
    "__pycache__/ignored.pyc": b"bytecode",
}

for relative_name, content in files_to_create.items():
    path = PROJECT_ROOT / relative_name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(content)

print(PROJECT_ROOT)

C:\Users\user1\AppData\Local\Temp\py312_walk_demo_6bgkahm4


## Problem 14 — Project inventory with pruning

Implement `inventory_tree(root, *, excluded_dirs)`.

Return:

```python
{
    "files": total_file_count,
    "bytes": total_bytes,
    "extensions": {".py": 3, ".md": 2, ...},
}
```

Requirements:

- use `Path.walk(top_down=True)`;
- prune excluded directory names by mutating `dirnames[:]`;
- extensionless files use `"<none>"`;
- normalize extensions to lowercase;
- count only regular files;
- return extensions sorted by extension.

### Solution 14

In [32]:
class Inventory(TypedDict):
    files: int
    bytes: int
    extensions: dict[str, int]


def inventory_tree(
    root: Path,
    *,
    excluded_dirs: Iterable[str] = (),
) -> Inventory:
    root = Path(root)
    excluded = set(excluded_dirs)

    total_files = 0
    total_bytes = 0
    extensions: Counter[str] = Counter()

    for directory, dirnames, filenames in root.walk(top_down=True):
        dirnames[:] = [
            dirname
            for dirname in dirnames
            if dirname not in excluded
        ]

        for filename in filenames:
            path = directory / filename
            if not path.is_file():
                continue

            total_files += 1
            total_bytes += path.stat().st_size
            extension = path.suffix.lower() or "<none>"
            extensions[extension] += 1

    return {
        "files": total_files,
        "bytes": total_bytes,
        "extensions": dict(sorted(extensions.items())),
    }


inventory_tree(PROJECT_ROOT, excluded_dirs={".git", "__pycache__"})

{'files': 6, 'bytes': 82, 'extensions': {'.bin': 1, '.md': 2, '.py': 3}}

In [33]:
inventory = inventory_tree(
    PROJECT_ROOT,
    excluded_dirs={".git", "__pycache__"},
)

check_equal(inventory["files"], 6)
check_equal(
    inventory["extensions"],
    {".bin": 1, ".md": 2, ".py": 3},
)
check_equal(
    inventory["bytes"],
    sum(
        len(content)
        for name, content in files_to_create.items()
        if not name.startswith(".git/")
        and not name.startswith("__pycache__/")
    ),
)

## Problem 15 — Find duplicate files by content

Implement `find_duplicate_files`.

Requirements:

1. Traverse with `Path.walk`.
2. Prune excluded directories.
3. First group files by size.
4. Hash only size groups containing at least two files.
5. Use SHA-256 and chunked reads.
6. Return only true duplicate groups.
7. Sort paths inside each group and sort groups deterministically.

### Solution 15

In [34]:
def sha256_file(path: Path, *, chunk_size: int = 64 * 1024) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)

    return digest.hexdigest()


def find_duplicate_files(
    root: Path,
    *,
    excluded_dirs: Iterable[str] = (),
) -> list[tuple[Path, ...]]:
    root = Path(root)
    excluded = set(excluded_dirs)
    by_size: dict[int, list[Path]] = defaultdict(list)

    for directory, dirnames, filenames in root.walk(top_down=True):
        dirnames[:] = [
            dirname
            for dirname in dirnames
            if dirname not in excluded
        ]

        for filename in filenames:
            path = directory / filename
            if path.is_file():
                by_size[path.stat().st_size].append(path)

    by_digest: dict[tuple[int, str], list[Path]] = defaultdict(list)

    for size, paths in by_size.items():
        if len(paths) < 2:
            continue
        for path in paths:
            by_digest[(size, sha256_file(path))].append(path)

    duplicate_groups = [
        tuple(sorted(paths))
        for paths in by_digest.values()
        if len(paths) > 1
    ]

    return sorted(
        duplicate_groups,
        key=lambda group: tuple(str(path) for path in group),
    )


duplicates = find_duplicate_files(
    PROJECT_ROOT,
    excluded_dirs={".git", "__pycache__"},
)
duplicates

[(WindowsPath('C:/Users/user1/AppData/Local/Temp/py312_walk_demo_6bgkahm4/docs/guide.md'),
  WindowsPath('C:/Users/user1/AppData/Local/Temp/py312_walk_demo_6bgkahm4/README.md'))]

In [35]:
check_equal(len(duplicates), 1)
check_equal(
    {path.relative_to(PROJECT_ROOT).as_posix() for path in duplicates[0]},
    {"README.md", "docs/guide.md"},
)

### Clean up the temporary project

In [36]:
shutil.rmtree(PROJECT_ROOT)
assert not PROJECT_ROOT.exists()

# 8. PEP 688 — Python-level buffer protocol

Python 3.12 allows pure Python classes to implement the buffer protocol with:

- `__buffer__(self, flags)`;
- optional `__release_buffer__(self, buffer)`.

This enables `memoryview(custom_object)` without a C extension.

The `collections.abc.Buffer` ABC can be used for runtime interface checks.

## Problem 16 — Mutable packet with zero-copy access

Implement `MutablePacket`.

Requirements:

- store data in a private `bytearray`;
- expose the data through `__buffer__`;
- count active and released views;
- support `len(packet)`;
- expose a `checksum()` equal to the sum of bytes modulo 256;
- allow mutation through `memoryview(packet)`;
- reject non-bytes-like constructor input naturally via `bytearray`.

### Solution 16

In [37]:
class MutablePacket:
    def __init__(self, data: bytes | bytearray | memoryview) -> None:
        self._data = bytearray(data)
        self.active_views = 0
        self.released_views = 0

    def __buffer__(self, flags: int) -> memoryview:
        self.active_views += 1
        return memoryview(self._data)

    def __release_buffer__(self, buffer: memoryview) -> None:
        self.active_views -= 1
        self.released_views += 1

    def __len__(self) -> int:
        return len(self._data)

    def checksum(self) -> int:
        return sum(self._data) % 256

    def to_bytes(self) -> bytes:
        return bytes(self._data)


packet = MutablePacket(b"ABC")
print("is Buffer:", isinstance(packet, Buffer))

view = memoryview(packet)
view[1] = ord("Z")
print(packet.to_bytes(), packet.checksum())
view.release()

print(
    "active_views=", packet.active_views,
    "released_views=", packet.released_views,
)

is Buffer: True
b'AZC' 222
active_views= 0 released_views= 1


In [38]:
check_equal(isinstance(packet, Buffer), True)
check_equal(packet.to_bytes(), b"AZC")
check_equal(len(packet), 3)
check_equal(packet.checksum(), (ord("A") + ord("Z") + ord("C")) % 256)
check_equal(packet.active_views, 0)
check_equal(packet.released_views, 1)

## Extra buffer example — Read-only export

A class can expose a read-only view even when its internal storage is mutable.

In [39]:
class ReadOnlyPacket:
    def __init__(self, data: bytes | bytearray | memoryview) -> None:
        self._data = bytearray(data)

    def __buffer__(self, flags: int) -> memoryview:
        return memoryview(self._data).toreadonly()

    def to_bytes(self) -> bytes:
        return bytes(self._data)


readonly_packet = ReadOnlyPacket(b"safe")
readonly_view = memoryview(readonly_packet)

check_equal(readonly_view.readonly, True)
check_equal(bytes(readonly_view), b"safe")
check_raises(TypeError, readonly_view.__setitem__, 0, ord("S"))
readonly_view.release()

# 9. PEP 669 — Low-impact monitoring with `sys.monitoring`

`sys.monitoring` is designed for debuggers, profilers, coverage tools, optimizers, and observability tooling.

Best practices:

- claim a tool ID before use;
- register only callbacks you need;
- prefer local events when monitoring selected code objects;
- disable events and unregister callbacks during cleanup;
- free the tool ID;
- use `try/finally` so instrumentation does not leak into later code.

## Problem 17 — Count recursive calls and returns

Implement `profile_function_calls(fn, *args, **kwargs)`.

Return:

```python
(result, {"starts": ..., "returns": ..., "max_depth": ...})
```

Requirements:

- monitor only `fn.__code__`;
- use `PY_START` and `PY_RETURN`;
- support recursion;
- always clean up;
- choose an available tool ID from 3 or 4;
- raise a helpful error if neither ID is available.

### Solution 17

In [40]:
class CallProfile(TypedDict):
    starts: int
    returns: int
    max_depth: int


def _claim_monitoring_tool(name: str) -> int:
    for tool_id in (3, 4):
        if sys.monitoring.get_tool(tool_id) is None:
            sys.monitoring.use_tool_id(tool_id, name)
            return tool_id
    raise RuntimeError("no free sys.monitoring tool ID among 3 and 4")


def profile_function_calls[T](
    fn: Callable[..., T],
    /,
    *args: object,
    **kwargs: object,
) -> tuple[T, CallProfile]:
    tool_id = _claim_monitoring_tool("advanced-notebook-profiler")
    events = sys.monitoring.events

    starts = 0
    returns = 0
    depth = 0
    max_depth = 0

    def on_start(code: object, instruction_offset: int) -> None:
        nonlocal starts, depth, max_depth
        starts += 1
        depth += 1
        max_depth = max(max_depth, depth)

    def on_return(
        code: object,
        instruction_offset: int,
        retval: object,
    ) -> None:
        nonlocal returns, depth
        returns += 1
        depth -= 1

    try:
        sys.monitoring.register_callback(tool_id, events.PY_START, on_start)
        sys.monitoring.register_callback(tool_id, events.PY_RETURN, on_return)
        sys.monitoring.set_local_events(
            tool_id,
            fn.__code__,
            events.PY_START | events.PY_RETURN,
        )

        result = fn(*args, **kwargs)

        return result, {
            "starts": starts,
            "returns": returns,
            "max_depth": max_depth,
        }
    finally:
        sys.monitoring.set_local_events(
            tool_id,
            fn.__code__,
            events.NO_EVENTS,
        )
        sys.monitoring.register_callback(tool_id, events.PY_START, None)
        sys.monitoring.register_callback(tool_id, events.PY_RETURN, None)
        sys.monitoring.free_tool_id(tool_id)


def factorial_recursive(n: int) -> int:
    if n < 0:
        raise ValueError("n must be non-negative")
    if n < 2:
        return 1
    return n * factorial_recursive(n - 1)


factorial_result, factorial_profile = profile_function_calls(
    factorial_recursive,
    6,
)

print(factorial_result, factorial_profile)

720 {'starts': 6, 'returns': 6, 'max_depth': 6}


In [41]:
check_equal(factorial_result, 720)
check_equal(
    factorial_profile,
    {"starts": 6, "returns": 6, "max_depth": 6},
)
check_equal(sys.monitoring.get_tool(3), None)
check_equal(sys.monitoring.get_tool(4), None)

## Monitoring edge case — exceptions and cleanup

`PY_RETURN` does not fire for a frame that exits by exception. Our profiler still cleans up because resource management is in `finally`.

In [42]:
def fail_recursively(n: int) -> None:
    if n == 0:
        raise RuntimeError("boom")
    fail_recursively(n - 1)


check_raises(
    RuntimeError,
    profile_function_calls,
    fail_recursively,
    3,
    contains="boom",
)
check_equal(sys.monitoring.get_tool(3), None)
check_equal(sys.monitoring.get_tool(4), None)

# 10. Migration checks for Python 3.12

A Python upgrade is not only about new features. It can also remove or deprecate old APIs.

Selected Python 3.12 concerns include:

- `distutils` was removed from the standard library;
- `datetime.datetime.utcnow()` and `utcfromtimestamp()` are deprecated in favor of timezone-aware alternatives;
- bitwise inversion of booleans (`~True`, `~False`) is deprecated.

The next exercise builds a small static audit tool. It is intentionally conservative and does not attempt full type inference.

## Problem 18 — AST-based migration auditor

Implement `audit_python_312(source)`.

Detect:

1. `import distutils`;
2. `import distutils.command`;
3. `from distutils... import ...`;
4. calls whose attribute name is `utcnow` or `utcfromtimestamp`;
5. literal `~True` or `~False`.

Return sorted findings as `(line_number, code, message)` tuples.

Codes:

- `PY312001` — removed `distutils`;
- `PY312002` — deprecated naive UTC datetime helper;
- `PY312003` — deprecated boolean bitwise inversion.

### Solution 18

In [43]:
type AuditFinding = tuple[int, str, str]


class Python312MigrationVisitor(ast.NodeVisitor):
    def __init__(self) -> None:
        self.findings: list[AuditFinding] = []

    def add(self, node: ast.AST, code: str, message: str) -> None:
        self.findings.append((node.lineno, code, message))

    @override
    def visit_Import(self, node: ast.Import) -> None:
        for alias in node.names:
            if alias.name == "distutils" or alias.name.startswith("distutils."):
                self.add(
                    node,
                    "PY312001",
                    f"'{alias.name}' was removed from the standard library",
                )
        self.generic_visit(node)

    @override
    def visit_ImportFrom(self, node: ast.ImportFrom) -> None:
        module = node.module or ""
        if module == "distutils" or module.startswith("distutils."):
            self.add(
                node,
                "PY312001",
                f"'{module}' was removed from the standard library",
            )
        self.generic_visit(node)

    @override
    def visit_Call(self, node: ast.Call) -> None:
        if (
            isinstance(node.func, ast.Attribute)
            and node.func.attr in {"utcnow", "utcfromtimestamp"}
        ):
            self.add(
                node,
                "PY312002",
                (
                    f"'{node.func.attr}' is deprecated; "
                    "prefer timezone-aware UTC datetime construction"
                ),
            )
        self.generic_visit(node)

    @override
    def visit_UnaryOp(self, node: ast.UnaryOp) -> None:
        if (
            isinstance(node.op, ast.Invert)
            and isinstance(node.operand, ast.Constant)
            and isinstance(node.operand.value, bool)
        ):
            self.add(
                node,
                "PY312003",
                "bitwise inversion of bool is deprecated; use 'not' for logic",
            )
        self.generic_visit(node)


def audit_python_312(source: str) -> list[AuditFinding]:
    tree = ast.parse(source)
    visitor = Python312MigrationVisitor()
    visitor.visit(tree)
    return sorted(visitor.findings)


legacy_source = '''
import distutils
from distutils.command.build import build
from datetime import datetime

stamp = datetime.utcnow()
other = datetime.utcfromtimestamp(0)
flag = ~True
'''

audit_python_312(legacy_source)

[(2, 'PY312001', "'distutils' was removed from the standard library"),
 (3,
  'PY312001',
  "'distutils.command.build' was removed from the standard library"),
 (6,
  'PY312002',
  "'utcnow' is deprecated; prefer timezone-aware UTC datetime construction"),
 (7,
  'PY312002',
  "'utcfromtimestamp' is deprecated; prefer timezone-aware UTC datetime construction"),
 (8,
  'PY312003',
  "bitwise inversion of bool is deprecated; use 'not' for logic")]

In [44]:
migration_findings = audit_python_312(legacy_source)

check_equal(
    [code for _, code, _ in migration_findings],
    [
        "PY312001",
        "PY312001",
        "PY312002",
        "PY312002",
        "PY312003",
    ],
)
check_equal(
    [line for line, _, _ in migration_findings],
    [2, 3, 6, 7, 8],
)
check_equal(audit_python_312("from pathlib import Path\nvalue = not True\n"), [])
check_raises(SyntaxError, audit_python_312, "def broken(:")

SyntaxError('invalid syntax', ('<unknown>', 1, 12, 'def broken(:\n', 1, 13))

# 11. Capstone — Project analytics pipeline

This capstone combines several Python 3.12 features:

- PEP 695 generic helper syntax;
- `Path.walk`;
- `itertools.batched`;
- `math.sumprod`;
- flexible f-strings;
- precise result schemas;
- deterministic testing.

## Problem 19 — Scan, rank, batch, and report files

Build a project report that:

1. walks a directory while excluding selected directory names;
2. collects each regular file's relative path, suffix, and size;
3. assigns an extension weight;
4. calculates weighted size as `size * weight`;
5. ranks files by weighted size descending, then path;
6. divides ranked files into batches;
7. reports total size and total weighted size;
8. renders a deterministic multi-line text report.

Use a generic `top_n` helper with PEP 695 syntax.

### Solution 19

In [45]:
@dataclass(frozen=True, slots=True)
class FileMetric:
    relative_path: str
    extension: str
    size: int
    weight: float

    @property
    def weighted_size(self) -> float:
        return self.size * self.weight


class ProjectReport(TypedDict):
    files: tuple[FileMetric, ...]
    total_size: int
    total_weighted_size: float
    batches: tuple[tuple[FileMetric, ...], ...]


def top_n[T](
    items: Iterable[T],
    n: int,
    *,
    key: Callable[[T], object],
) -> list[T]:
    if n < 0:
        raise ValueError("n must be non-negative")
    return sorted(items, key=key)[:n]


def scan_file_metrics(
    root: Path,
    *,
    extension_weights: Mapping[str, float],
    excluded_dirs: Iterable[str] = (),
) -> list[FileMetric]:
    root = Path(root)
    excluded = set(excluded_dirs)
    metrics: list[FileMetric] = []

    for directory, dirnames, filenames in root.walk(top_down=True):
        dirnames[:] = [
            dirname
            for dirname in dirnames
            if dirname not in excluded
        ]

        for filename in filenames:
            path = directory / filename
            if not path.is_file():
                continue

            extension = path.suffix.lower() or "<none>"
            weight = float(extension_weights.get(extension, 1.0))

            if weight < 0 or not math.isfinite(weight):
                raise ValueError(
                    f"invalid weight {weight!r} for extension {extension!r}"
                )

            metrics.append(
                FileMetric(
                    relative_path=path.relative_to(root).as_posix(),
                    extension=extension,
                    size=path.stat().st_size,
                    weight=weight,
                )
            )

    return metrics


def build_project_report(
    root: Path,
    *,
    extension_weights: Mapping[str, float],
    batch_size: int = 3,
    excluded_dirs: Iterable[str] = (),
) -> ProjectReport:
    if batch_size <= 0:
        raise ValueError("batch_size must be positive")

    metrics = scan_file_metrics(
        root,
        extension_weights=extension_weights,
        excluded_dirs=excluded_dirs,
    )

    ranked = tuple(
        sorted(
            metrics,
            key=lambda item: (-item.weighted_size, item.relative_path),
        )
    )

    sizes = tuple(metric.size for metric in ranked)
    weights = tuple(metric.weight for metric in ranked)

    return {
        "files": ranked,
        "total_size": sum(sizes),
        "total_weighted_size": math.sumprod(sizes, weights),
        "batches": tuple(batched(ranked, batch_size)),
    }


def render_project_report(report: ProjectReport, *, top: int = 5) -> str:
    top_files = top_n(
        report["files"],
        top,
        key=lambda item: (-item.weighted_size, item.relative_path),
    )

    lines = [
        "PROJECT REPORT",
        f"files={len(report["files"])}",
        f"batches={len(report["batches"])}",
        f"total_size={report["total_size"]} bytes",
        f"weighted_size={report["total_weighted_size"]:.2f}",
        "top_files:",
    ]

    lines.extend(
        f"  {rank:>2}. {metric.relative_path} "
        f"size={metric.size} weight={metric.weight:.2f} "
        f"weighted={metric.weighted_size:.2f}"
        for rank, metric in enumerate(top_files, start=1)
    )

    return "\n".join(lines)

In [46]:
with tempfile.TemporaryDirectory(prefix="py312_capstone_") as temp_dir:
    root = Path(temp_dir)

    capstone_files = {
        "src/main.py": b"print('main')\n",
        "src/model.py": b"class Model: pass\n",
        "assets/logo.bin": bytes(range(32)),
        "README.md": b"# Demo\n",
        ".git/config": b"ignore me",
    }

    for relative_name, content in capstone_files.items():
        path = root / relative_name
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(content)

    report = build_project_report(
        root,
        extension_weights={
            ".py": 2.0,
            ".md": 1.25,
            ".bin": 0.5,
        },
        batch_size=2,
        excluded_dirs={".git"},
    )

    rendered_report = render_project_report(report, top=3)
    print(rendered_report)

    check_equal(len(report["files"]), 4)
    check_equal(len(report["batches"]), 2)
    check_equal(
        report["total_size"],
        sum(
            len(content)
            for name, content in capstone_files.items()
            if not name.startswith(".git/")
        ),
    )

    expected_weighted = (
        len(capstone_files["src/main.py"]) * 2.0
        + len(capstone_files["src/model.py"]) * 2.0
        + len(capstone_files["README.md"]) * 1.25
        + len(capstone_files["assets/logo.bin"]) * 0.5
    )
    check_close(report["total_weighted_size"], expected_weighted)
    assert "PROJECT REPORT" in rendered_report
    assert "top_files:" in rendered_report

PROJECT REPORT
files=4
batches=2
total_size=71 bytes
weighted_size=88.75
top_files:
   1. src/model.py size=18 weight=2.00 weighted=36.00
   2. src/main.py size=14 weight=2.00 weighted=28.00
   3. assets/logo.bin size=32 weight=0.50 weighted=16.00


# 12. Further advanced exercises

Use these as extensions. They intentionally do not include full solutions so that the notebook's solved problems can serve as patterns.

1. **F-string diagnostics:** create a format-string compiler that validates a user-selected width and precision before rendering.
2. **Generic graph:** implement `DirectedGraph[NodeT: Hashable]` with breadth-first and depth-first traversal.
3. **Variadic generics:** model matrix dimensions with a `TypeVarTuple` using PEP 695 syntax.
4. **Typed configuration:** combine several `TypedDict` schemas and validate a loaded TOML configuration.
5. **Comprehension tracing:** compare trace events for comprehensions on Python 3.11 and 3.12.
6. **Batch retry engine:** process batches, retry transient failures, and preserve idempotency keys.
7. **Numerical stability:** compare `sum(a*b for ...)`, `math.fsum`, and `math.sumprod` on adversarial floats.
8. **Filesystem synchronizer:** use `Path.walk` to compute additions, removals, and modified files between directory trees.
9. **Buffer-backed parser:** parse binary packet headers through a read-only `memoryview` without copying.
10. **Coverage prototype:** use `sys.monitoring.events.LINE` locally to collect executed line numbers.
11. **Migration linter:** expand the AST auditor with configurable rules and source excerpts.
12. **Capstone export:** serialize the project report to JSON and CSV while preserving deterministic ordering.

# Summary

Python 3.12 is more than a syntax release.

The changes in this notebook support:

- clearer generic APIs;
- more expressive string construction;
- faster comprehensions;
- safer iterable batching;
- validated numerical dot products;
- object-oriented filesystem traversal;
- pure Python buffer exporters;
- lower-impact execution instrumentation;
- more deliberate migrations from legacy APIs.

The strongest practice is not to use every new feature. It is to choose features that make invariants explicit, reduce accidental complexity, and improve testability.